In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# Fixed reporting date makes the synthetic cleaning reproducible.
AS_OF_DATE = pd.Timestamp("2026-09-22")

# Change this to your actual folder.
BASE_DIR = Path(
    r"C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets"
)

CLEAN_DIR = BASE_DIR / "cleaned"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder :", BASE_DIR)
print("Output folder:", CLEAN_DIR)

Input folder : C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets
Output folder: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned


In [4]:
FILES = {
    "students": BASE_DIR / "students.xlsx",
    "consultants": BASE_DIR / "consultants.xlsx",
    "marketing_campaigns": BASE_DIR / "marketing_campaigns.xlsx",
    "student_journeys": BASE_DIR / "student_journey_consistent.xlsx",
    "university_applications": BASE_DIR / "university_applications_consistent.xlsx",
    "visa_applications": BASE_DIR / "visa_applications_consistent.xlsx",
    "case_events": BASE_DIR / "case_events.xlsx"
}

for name, path in FILES.items():
    print(f"{name:25} -> {path.exists()} -> {path.name}")

students                  -> True -> students.xlsx
consultants               -> True -> consultants.xlsx
marketing_campaigns       -> True -> marketing_campaigns.xlsx
student_journeys          -> True -> student_journey_consistent.xlsx
university_applications   -> True -> university_applications_consistent.xlsx
visa_applications         -> True -> visa_applications_consistent.xlsx
case_events               -> False -> case_events.xlsx


In [5]:
NULL_VALUES = {
    "",
    "-",
    "null",
    "NULL",
    "None",
    "none",
    "N/A",
    "n/a",
    "NA",
    "nan"
}


def standardize_columns(df):
    """
    Converts column names such as:
    'Lead Date' -> 'lead_date'
    'Phone-Number' -> 'phone_number'
    """
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )

    return df


def clean_text(series):
    """
    Removes whitespace and converts fake null values to actual missing values.
    """
    s = series.astype("string").str.strip()

    return s.replace(list(NULL_VALUES), pd.NA)


def clean_id(series):
    """
    Converts IDs safely to strings.

    Examples:
        1       -> "1"
        1.0     -> "1"
        "-"     -> <NA>
    """

    def convert(value):
        if pd.isna(value):
            return pd.NA

        value = str(value).strip()

        if value in NULL_VALUES:
            return pd.NA

        # Convert strings such as "5.0" to "5"
        if re.fullmatch(r"\d+\.0", value):
            value = value[:-2]

        return value

    return series.apply(convert).astype("string")


def parse_mixed_date(series):
    """
    Handles both:
      - Excel serial numbers such as 45756
      - Strings such as 03/24/2025
    """

    result = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    numeric_values = pd.to_numeric(series, errors="coerce")
    numeric_mask = numeric_values.notna()

    # Excel dates use 1899-12-30 as practical origin
    result.loc[numeric_mask] = pd.to_datetime(
        numeric_values.loc[numeric_mask],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )

    text_mask = (~numeric_mask) & series.notna()

    result.loc[text_mask] = pd.to_datetime(
        series.loc[text_mask],
        errors="coerce"
    )

    return result.dt.normalize()


def show_audit(df, name, primary_key=None):
    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isna().sum())

    if primary_key:
        print(
            f"\nDuplicate {primary_key}:",
            df[primary_key].duplicated().sum()
        )

    print("\nData types:")
    print(df.dtypes)

    display(df.head())

In [6]:
students = standardize_columns(
    pd.read_excel(FILES["students"])
)

consultants = standardize_columns(
    pd.read_excel(FILES["consultants"])
)

campaigns = standardize_columns(
    pd.read_excel(FILES["marketing_campaigns"])
)

journeys = standardize_columns(
    pd.read_excel(FILES["student_journeys"])
)

applications = standardize_columns(
    pd.read_excel(FILES["university_applications"])
)

visas = standardize_columns(
    pd.read_excel(FILES["visa_applications"])
)


In [7]:
datasets = {
    "students": students,
    "consultants": consultants,
    "marketing_campaigns": campaigns,
    "student_journeys": journeys,
    "university_applications": applications,
    "visa_applications": visas
}

for name, df in datasets.items():
    print(name, df.shape)

students (1000, 7)
consultants (5, 5)
marketing_campaigns (30, 7)
student_journeys (1000, 8)
university_applications (1118, 5)
visa_applications (289, 4)


In [8]:
students_clean = students.copy()

EXPECTED_STUDENT_COLUMNS = [
    "s_id",
    "first_name",
    "last_name",
    "dob",
    "nationality",
    "email",
    "phone_number"
]

students_clean = students_clean[
    EXPECTED_STUDENT_COLUMNS
].copy()

# IDs
students_clean["s_id"] = clean_id(students_clean["s_id"])

# Names
students_clean["first_name"] = (
    clean_text(students_clean["first_name"])
    .str.title()
)

students_clean["last_name"] = (
    clean_text(students_clean["last_name"])
    .str.title()
)

# DOB
students_clean["dob"] = parse_mixed_date(
    students_clean["dob"]
)

# This agency's target population is Burmese students
students_clean["nationality"] = "Myanmar"

In [9]:
student_age = (
    (AS_OF_DATE - students_clean["dob"]).dt.days / 365.25
)

print(student_age.describe())

invalid_age = (
    (student_age < 17) |
    (student_age > 45)
)

print("Students outside expected age range:", invalid_age.sum())

display(
    students_clean.loc[
        invalid_age,
        ["s_id", "first_name", "last_name", "dob"]
    ].head(20)
)

count    1000.000000
mean       26.209238
std         3.236578
min        20.733744
25%        23.386037
50%        26.183436
75%        29.071184
max        31.687885
Name: dob, dtype: float64
Students outside expected age range: 0


,s_id,first_name,last_name,dob


In [10]:
def safe_email_part(value):
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9]", "", value)
    return value


students_clean["email"] = students_clean.apply(
    lambda row:
        f"{safe_email_part(row['first_name'])}."
        f"{safe_email_part(row['last_name'])}."
        f"{row['s_id']}@student.example",
    axis=1
)

In [11]:
students_clean["email"].duplicated().sum()

np.int64(0)

In [12]:
students_clean["phone_number"] = [
    f"+95-9-000-{i:06d}"
    for i in range(1, len(students_clean) + 1)
]

In [13]:
show_audit(
    students_clean,
    "students_clean",
    "s_id"
)


STUDENTS_CLEAN
Shape: (1000, 7)

Missing values:
s_id            0
first_name      0
last_name       0
dob             0
nationality     0
email           0
phone_number    0
dtype: int64

Duplicate s_id: 0

Data types:
s_id                    string
first_name              string
last_name               string
dob             datetime64[ns]
nationality                str
email                      str
phone_number               str
dtype: object


,s_id,first_name,last_name,dob,nationality,email,phone_number
0,1,Oralla,Stoter,1995-07-20,Myanmar,oralla.stoter.1@student.example,+95-9-000-000001
1,2,Worden,Dawkins,1998-09-07,Myanmar,worden.dawkins.2@student.example,+95-9-000-000002
2,3,Dale,Gorrie,2005-04-04,Myanmar,dale.gorrie.3@student.example,+95-9-000-000003
3,4,Trudy,Wyne,1997-06-17,Myanmar,trudy.wyne.4@student.example,+95-9-000-000004
4,5,Nap,Ugo,2004-11-16,Myanmar,nap.ugo.5@student.example,+95-9-000-000005


In [14]:
consultants_clean = consultants.copy()

CONSULTANT_COLUMNS = [
    "c_id",
    "consultant_name",
    "consultant_role",
    "email",
    "phone_number"
]

consultants_clean = consultants_clean[
    CONSULTANT_COLUMNS
].copy()

consultants_clean["c_id"] = clean_id(
    consultants_clean["c_id"]
)

consultants_clean["consultant_name"] = (
    clean_text(consultants_clean["consultant_name"])
    .str.title()
)

roles = [
    "Senior Education Consultant",
    "Education Consultant",
    "Admissions Consultant",
    "Visa Consultant",
    "Team Lead"
]

consultants_clean["consultant_role"] = [
    roles[i % len(roles)]
    for i in range(len(consultants_clean))
]

In [15]:
consultants_clean["email"] = consultants_clean.apply(
    lambda row:
        f"{safe_email_part(row['consultant_name'])}@agency.example",
    axis=1
)

consultants_clean["phone_number"] = [
    f"+353-1-555-{1000 + i}"
    for i in range(len(consultants_clean))
]

In [16]:
show_audit(
    consultants_clean,
    "consultants_clean",
    "c_id"
)


CONSULTANTS_CLEAN
Shape: (5, 5)

Missing values:
c_id               0
consultant_name    0
consultant_role    0
email              0
phone_number       0
dtype: int64

Duplicate c_id: 0

Data types:
c_id               string
consultant_name    string
consultant_role       str
email                 str
phone_number          str
dtype: object


,c_id,consultant_name,consultant_role,email,phone_number
0,1,Marius Itshak,Senior Education Consultant,mariusitshak@agency.example,+353-1-555-1000
1,2,Hernando Bamford,Education Consultant,hernandobamford@agency.example,+353-1-555-1001
2,3,Shay Hastie,Admissions Consultant,shayhastie@agency.example,+353-1-555-1002
3,4,Damaris Wilce,Visa Consultant,damariswilce@agency.example,+353-1-555-1003
4,5,Janelle Devonald,Team Lead,janelledevonald@agency.example,+353-1-555-1004


In [17]:
campaigns_clean = campaigns.copy()

CAMPAIGN_COLUMNS = [
    "mc_id",
    "campaign_name",
    "platform",
    "start_date",
    "end_date",
    "budget_eur",
    "actual_spend_eur"
]

campaigns_clean = campaigns_clean[
    CAMPAIGN_COLUMNS
].copy()

campaigns_clean["mc_id"] = clean_id(
    campaigns_clean["mc_id"]
)

campaigns_clean["campaign_name"] = clean_text(
    campaigns_clean["campaign_name"]
)

campaigns_clean["platform"] = clean_text(
    campaigns_clean["platform"]
)

campaigns_clean["start_date"] = parse_mixed_date(
    campaigns_clean["start_date"]
)

campaigns_clean["end_date"] = parse_mixed_date(
    campaigns_clean["end_date"]
)

campaigns_clean["budget_eur"] = pd.to_numeric(
    campaigns_clean["budget_eur"],
    errors="coerce"
)

campaigns_clean["actual_spend_eur"] = pd.to_numeric(
    campaigns_clean["actual_spend_eur"],
    errors="coerce"
)

In [18]:
reversed_dates = (
    campaigns_clean["end_date"] <
    campaigns_clean["start_date"]
)

print(
    "Campaigns with reversed dates:",
    reversed_dates.sum()
)

start_copy = campaigns_clean.loc[
    reversed_dates,
    "start_date"
].copy()

campaigns_clean.loc[
    reversed_dates,
    "start_date"
] = campaigns_clean.loc[
    reversed_dates,
    "end_date"
]

campaigns_clean.loc[
    reversed_dates,
    "end_date"
] = start_copy

Campaigns with reversed dates: 13


In [19]:
# Negative values make no sense
campaigns_clean["budget_eur"] = (
    campaigns_clean["budget_eur"].abs()
)

campaigns_clean["actual_spend_eur"] = (
    campaigns_clean["actual_spend_eur"].abs()
)

# Fill missing budget using median
campaigns_clean["budget_eur"] = (
    campaigns_clean["budget_eur"]
    .fillna(campaigns_clean["budget_eur"].median())
)

# Missing actual spend = 90% of budget
campaigns_clean["actual_spend_eur"] = (
    campaigns_clean["actual_spend_eur"]
    .fillna(campaigns_clean["budget_eur"] * 0.90)
)

# Allow up to 10% campaign overspend
campaigns_clean["actual_spend_eur"] = np.minimum(
    campaigns_clean["actual_spend_eur"],
    campaigns_clean["budget_eur"] * 1.10
)

campaigns_clean["budget_eur"] = (
    campaigns_clean["budget_eur"].round(2)
)

campaigns_clean["actual_spend_eur"] = (
    campaigns_clean["actual_spend_eur"].round(2)
)

In [20]:
show_audit(
    campaigns_clean,
    "marketing_campaigns_clean",
    "mc_id"
)


MARKETING_CAMPAIGNS_CLEAN
Shape: (30, 7)

Missing values:
mc_id               0
campaign_name       0
platform            0
start_date          0
end_date            0
budget_eur          0
actual_spend_eur    0
dtype: int64

Duplicate mc_id: 0

Data types:
mc_id                       string
campaign_name               string
platform                    string
start_date          datetime64[ns]
end_date            datetime64[ns]
budget_eur                 float64
actual_spend_eur           float64
dtype: object


,mc_id,campaign_name,platform,start_date,end_date,budget_eur,actual_spend_eur
0,1,Study in Ireland Webinar,Instagram,2027-01-05,2027-06-09,303.89,210.45
1,2,Study in Ireland Webinar,Facebook,2026-07-16,2027-12-17,105.27,115.80
2,3,January 2027 Early Application,YouTube,2025-10-03,2026-11-14,304.80,283.32
3,4,January 2027 Early Application,YouTube,2025-12-23,2026-07-29,412.18,22.18
4,5,January 2027 Early Application,Referral Campaign,2026-01-02,2026-12-03,233.83,257.21


In [21]:
journeys_clean = journeys.copy()

journeys_clean["sj_id"] = clean_id(
    journeys_clean["sj_id"]
)

journeys_clean["s_id"] = clean_id(
    journeys_clean["s_id"]
)

journeys_clean["mc_id"] = clean_id(
    journeys_clean["mc_id"]
)

journeys_clean["lead_date"] = parse_mixed_date(
    journeys_clean["lead_date"]
)

journeys_clean["lead_source"] = clean_text(
    journeys_clean["lead_source"]
)

journeys_clean["target_intake_term"] = clean_text(
    journeys_clean["target_intake_term"]
)

journeys_clean["current_stage"] = clean_text(
    journeys_clean["current_stage"]
)

journeys_clean["target_intake_year"] = pd.to_numeric(
    journeys_clean["target_intake_year"],
    errors="coerce"
).astype("Int64")

In [22]:
future_leads = (
    journeys_clean["lead_date"] > AS_OF_DATE
)

print("Future leads found:", future_leads.sum())

for counter, idx in enumerate(
    journeys_clean.index[future_leads]
):
    journeys_clean.loc[idx, "lead_date"] = (
        AS_OF_DATE -
        pd.Timedelta(days=(counter % 180))
    )

Future leads found: 421


In [23]:
rng = np.random.default_rng(42)

valid_consultants = (
    consultants_clean["c_id"]
    .dropna()
    .tolist()
)

if "c_id" not in journeys_clean.columns:
    journeys_clean["c_id"] = rng.choice(
        valid_consultants,
        size=len(journeys_clean)
    )
else:
    journeys_clean["c_id"] = clean_id(
        journeys_clean["c_id"]
    )

    invalid_consultant = (
        journeys_clean["c_id"].isna() |
        ~journeys_clean["c_id"].isin(valid_consultants)
    )

    journeys_clean.loc[
        invalid_consultant,
        "c_id"
    ] = rng.choice(
        valid_consultants,
        size=invalid_consultant.sum()
    )

In [24]:
valid_campaign_ids = set(
    campaigns_clean["mc_id"].dropna()
)

invalid_mc = (
    journeys_clean["mc_id"].notna() &
    ~journeys_clean["mc_id"].isin(valid_campaign_ids)
)

print(
    "Invalid campaign IDs:",
    invalid_mc.sum()
)

# Optional campaign relationship -> invalid becomes NULL
journeys_clean.loc[
    invalid_mc,
    "mc_id"
] = pd.NA

Invalid campaign IDs: 0


In [25]:
valid_campaign_ids = set(
    campaigns_clean["mc_id"].dropna()
)

invalid_mc = (
    journeys_clean["mc_id"].notna() &
    ~journeys_clean["mc_id"].isin(valid_campaign_ids)
)

print(
    "Invalid campaign IDs:",
    invalid_mc.sum()
)

# Optional campaign relationship -> invalid becomes NULL
journeys_clean.loc[
    invalid_mc,
    "mc_id"
] = pd.NA

Invalid campaign IDs: 0


In [26]:
platform_lookup = (
    campaigns_clean
    .set_index("mc_id")["platform"]
    .to_dict()
)

has_campaign = journeys_clean["mc_id"].notna()

journeys_clean.loc[
    has_campaign,
    "lead_source"
] = journeys_clean.loc[
    has_campaign,
    "mc_id"
].map(platform_lookup)

In [27]:
LEAD_SOURCE_MAP = {
    "google ads": "Google",
    "google": "Google",
    "student referral": "Student Referral",
    "referral": "Student Referral",
    "website": "Website",
    "tiktok": "TikTok",
    "facebook": "Facebook",
    "instagram": "Instagram",
    "walk-in": "Walk-in / Direct",
    "direct": "Walk-in / Direct"
}

journeys_clean["lead_source"] = journeys_clean[
    "lead_source"
].apply(
    lambda x:
        LEAD_SOURCE_MAP.get(
            str(x).lower(),
            x
        )
)

In [28]:
STAGE_ALIASES = {
    "new lead": "New Lead",
    "consultation scheduled": "Consultation Scheduled",
    "consultation completed": "Consultation Completed",
    "documents pending": "Documents Pending",
    "application in progress": "Application In Progress",
    "application submitted": "Application Submitted",
    "offer received": "Offer Received",
    "offer accepted": "Offer Accepted",
    "visa preparation": "Visa Preparation",
    "visa submitted": "Visa Submitted",
    "visa approved": "Visa Approved",
    "arrived in ireland": "Arrived in Ireland",
    "enrolled": "Enrolled",
    "lost": "Closed Lost",
    "closed lost": "Closed Lost",
    "withdrawn": "Withdrawn",
    "on hold": "On Hold"
}


def standardize_stage(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    return STAGE_ALIASES.get(
        value.lower(),
        value
    )


journeys_clean["current_stage"] = (
    journeys_clean["current_stage"]
    .apply(standardize_stage)
)

In [29]:
VALID_STAGES = [
    "New Lead",
    "Consultation Scheduled",
    "Consultation Completed",
    "Documents Pending",
    "Application In Progress",
    "Application Submitted",
    "Offer Received",
    "Offer Accepted",
    "Offer Rejected",
    "Visa Preparation",
    "Visa Submitted",
    "Visa Approved",
    "Arrived in Ireland",
    "Enrolled",
    "Closed Lost",
    "Withdrawn",
    "On Hold",
    "Closed"
    
]

unknown_stages = set(
    journeys_clean["current_stage"].dropna()
) - set(VALID_STAGES)

print("Unknown stages:", unknown_stages)

Unknown stages: set()


In [30]:
MIN_PROCESS_DAYS = {
    "New Lead": 0,
    "Consultation Scheduled": 3,
    "Consultation Completed": 7,
    "Documents Pending": 14,
    "Application In Progress": 25,
    "Application Submitted": 40,
    "Offer Received": 65,
    "Offer Accepted": 75,
    "Visa Preparation": 90,
    "Visa Submitted": 105,
    "Visa Approved": 130,
    "Arrived in Ireland": 145,
    "Enrolled": 150,
    "Closed Lost": 7,
    "Withdrawn": 7,
    "On Hold": 7
}


for idx, row in journeys_clean.iterrows():

    stage = row["current_stage"]
    lead_date = row["lead_date"]

    if pd.isna(lead_date):
        continue

    minimum_days = MIN_PROCESS_DAYS.get(stage, 0)

    latest_possible_lead = (
        AS_OF_DATE -
        pd.Timedelta(days=minimum_days)
    )

    if lead_date > latest_possible_lead:

        journeys_clean.loc[idx, "lead_date"] = (
            latest_possible_lead -
            pd.Timedelta(days=(idx % 30))
        )

In [31]:
INTAKE_MAP = {
    "january": "January",
    "may": "May",
    "september": "September"
}

journeys_clean["target_intake_term"] = (
    journeys_clean["target_intake_term"]
    .str.lower()
    .map(INTAKE_MAP)
)

TERM_MONTH = {
    "January": 1,
    "May": 5,
    "September": 9
}

In [32]:
def fix_intake_year(row):

    lead_date = row["lead_date"]
    term = row["target_intake_term"]
    original_year = row["target_intake_year"]

    if pd.isna(lead_date):
        return original_year

    if pd.isna(term):
        term = "September"

    lead_year = lead_date.year

    try:
        target_year = int(original_year)
    except:
        target_year = lead_year

    # No past intake
    target_year = max(
        target_year,
        lead_year
    )

    # Avoid unrealistic 3–4 year future targets
    target_year = min(
        target_year,
        lead_year + 2
    )

    target_date = pd.Timestamp(
        year=target_year,
        month=TERM_MONTH[term],
        day=1
    )

    # Give student at least ~60 days before intake
    if target_date < (
        lead_date +
        pd.Timedelta(days=60)
    ):
        target_year += 1

    return min(
        target_year,
        lead_year + 2
    )


journeys_clean["target_intake_year"] = (
    journeys_clean.apply(
        fix_intake_year,
        axis=1
    )
    .astype("Int64")
)

In [33]:
def derive_case_status(stage):

    if stage == "Closed Lost":
        return "Lost"

    if stage == "Withdrawn":
        return "Withdrawn"

    if stage == "On Hold":
        return "On Hold"

    if stage in [
        "Arrived in Ireland",
        "Enrolled"
    ]:
        return "Completed"

    return "Active"


journeys_clean["case_status"] = (
    journeys_clean["current_stage"]
    .apply(derive_case_status)
)

In [34]:
applications_clean = applications.copy()

applications_clean["app_id"] = clean_id(
    applications_clean["app_id"]
)

applications_clean["sj_id"] = clean_id(
    applications_clean["sj_id"]
)

applications_clean["university_name"] = clean_text(
    applications_clean["university_name"]
)

applications_clean["course_name"] = clean_text(
    applications_clean["course_name"]
)

applications_clean["application_status"] = clean_text(
    applications_clean["application_status"]
)

In [35]:
applications_clean = (
    applications_clean
    .drop_duplicates(
        subset=["app_id"],
        keep="first"
    )
    .copy()
)

In [36]:
valid_journeys = set(
    journeys_clean["sj_id"]
)

orphan_apps = (
    ~applications_clean["sj_id"]
    .isin(valid_journeys)
)

print(
    "Orphan applications removed:",
    orphan_apps.sum()
)

applications_clean = (
    applications_clean.loc[
        ~orphan_apps
    ]
    .copy()
)

Orphan applications removed: 0


In [37]:
APPLICATION_STATUS_MAP = {
    "draft": "Draft",
    "documents pending": "Documents Pending",
    "submitted": "Submitted",
    "under review": "Under Review",
    "offer issued": "Offer Issued",
    "conditional offer": "Offer Issued",
    "unconditional offer": "Offer Issued",
    "rejected": "Rejected",
    "withdrawn": "Withdrawn",
    "deferred": "Deferred"
}

applications_clean["application_status"] = (
    applications_clean["application_status"]
    .str.lower()
    .map(APPLICATION_STATUS_MAP)
)

In [38]:
applications_clean[
    "application_status"
].value_counts(dropna=False)

application_status
Offer Issued         583
Rejected             154
Documents Pending    144
Under Review         122
Submitted            115
Name: count, dtype: int64

In [39]:
APPLICATION_OUTCOME = {
    "Offer Issued": "Offer",
    "Rejected": "Rejected",
    "Withdrawn": "Withdrawn",
    "Deferred": "Deferred"
}

applications_clean["application_outcome"] = (
    applications_clean["application_status"]
    .map(APPLICATION_OUTCOME)
)

In [40]:
duplicate_programme_apps = (
    applications_clean.duplicated(
        subset=[
            "sj_id",
            "university_name",
            "course_name"
        ],
        keep="first"
    )
)

print(
    "Duplicate journey/university/course applications removed:",
    duplicate_programme_apps.sum()
)

applications_clean = (
    applications_clean.loc[
        ~duplicate_programme_apps
    ]
    .copy()
)

Duplicate journey/university/course applications removed: 0


In [41]:
visas_clean = visas.copy()

visas_clean["visa_id"] = clean_id(
    visas_clean["visa_id"]
)

visas_clean["sj_id"] = clean_id(
    visas_clean["sj_id"]
)

visas_clean["app_id"] = clean_id(
    visas_clean["app_id"]
)

visas_clean["visa_status"] = clean_text(
    visas_clean["visa_status"]
)

visas_clean = (
    visas_clean
    .drop_duplicates(
        subset=["visa_id"],
        keep="first"
    )
    .copy()
)

In [42]:
VISA_STATUS_MAP = {
    "not started": "Not Started",
    "preparing": "Preparing",
    "documents pending": "Documents Pending",
    "submitted": "Submitted",
    "waiting": "Pending Decision",
    "pending decision": "Pending Decision",
    "approved": "Approved",
    "refused": "Refused",
    "withdrawn": "Withdrawn"
}

visas_clean["visa_status"] = (
    visas_clean["visa_status"]
    .str.lower()
    .map(VISA_STATUS_MAP)
)

In [43]:
application_parent = (
    applications_clean
    .set_index("app_id")["sj_id"]
    .to_dict()
)

valid_application_ids = set(
    applications_clean["app_id"]
)

In [44]:
invalid_visa_application = (
    visas_clean["app_id"].notna() &
    ~visas_clean["app_id"].isin(
        valid_application_ids
    )
)

print(
    "Visa records with invalid app_id:",
    invalid_visa_application.sum()
)

Visa records with invalid app_id: 0


In [45]:
expected_journey = (
    visas_clean["app_id"]
    .map(application_parent)
)

journey_mismatch = (
    visas_clean["app_id"].notna() &
    expected_journey.notna() &
    (
        visas_clean["sj_id"] !=
        expected_journey
    )
)

print(
    "Visa/app journey mismatches corrected:",
    journey_mismatch.sum()
)

visas_clean.loc[
    journey_mismatch,
    "sj_id"
] = expected_journey.loc[
    journey_mismatch
]

Visa/app journey mismatches corrected: 0


In [46]:
VISA_OUTCOME_MAP = {
    "Approved": "Approved",
    "Refused": "Refused",
    "Withdrawn": "Withdrawn"
}

visas_clean["visa_outcome"] = (
    visas_clean["visa_status"]
    .map(VISA_OUTCOME_MAP)
)

In [47]:
rng = np.random.default_rng(42)
pending = visas_clean.index[visas_clean["visa_status"] == "Pending Decision"]
refuse  = rng.choice(pending, size=15, replace=False)

visas_clean.loc[refuse, "visa_status"]  = "Refused"
visas_clean.loc[refuse, "visa_outcome"] = "Refused"

print(visas_clean["visa_status"].value_counts())

visa_status
Approved            135
Preparing            74
Pending Decision     65
Refused              15
Name: count, dtype: int64


In [48]:
STAGE_RANK = {
    "New Lead": 1,
    "Consultation Scheduled": 2,
    "Consultation Completed": 3,
    "Documents Pending": 4,
    "Application In Progress": 5,
    "Application Submitted": 6,
    "Offer Received": 7,
    "Offer Accepted": 8,
    "Visa Preparation": 9,
    "Visa Submitted": 10,
    "Visa Approved": 11,
    "Arrived in Ireland": 12,
    "Enrolled": 13
}

RANK_STAGE = {
    value: key
    for key, value in STAGE_RANK.items()
}

In [49]:
APP_STAGE_RANK = {
    "Draft": 5,
    "Documents Pending": 5,
    "Submitted": 6,
    "Under Review": 6,
    "Offer Issued": 7,
    "Rejected": 6,
    "Withdrawn": 6,
    "Deferred": 6
}

applications_clean["evidence_rank"] = (
    applications_clean["application_status"]
    .map(APP_STAGE_RANK)
)

app_evidence = (
    applications_clean
    .groupby("sj_id")["evidence_rank"]
    .max()
    .to_dict()
)

In [50]:
VISA_STAGE_RANK = {
    "Not Started": 8,
    "Preparing": 9,
    "Documents Pending": 9,
    "Submitted": 10,
    "Pending Decision": 10,
    "Approved": 11,
    "Refused": 10,
    "Withdrawn": 10
}

visas_clean["evidence_rank"] = (
    visas_clean["visa_status"]
    .map(VISA_STAGE_RANK)
)

visa_evidence = (
    visas_clean
    .groupby("sj_id")["evidence_rank"]
    .max()
    .to_dict()
)

In [51]:
TERMINAL_STAGES = {
    "Closed Lost",
    "Withdrawn",
    "On Hold"
}


def reconcile_current_stage(row):

    current_stage = row["current_stage"]
    sj_id = row["sj_id"]

    # Preserve terminal case states.
    if current_stage in TERMINAL_STAGES:
        return current_stage

    current_rank = STAGE_RANK.get(
        current_stage,
        1
    )

    app_rank = app_evidence.get(
        sj_id,
        0
    )

    visa_rank = visa_evidence.get(
        sj_id,
        0
    )

    final_rank = max(
        current_rank,
        app_rank,
        visa_rank
    )

    return RANK_STAGE.get(
        final_rank,
        current_stage
    )


journeys_clean["current_stage"] = (
    journeys_clean.apply(
        reconcile_current_stage,
        axis=1
    )
)

journeys_clean["case_status"] = (
    journeys_clean["current_stage"]
    .apply(derive_case_status)
)

In [52]:
campaign_leads = (
    journeys_clean[
        journeys_clean["mc_id"].notna()
    ]
    .groupby("mc_id")["lead_date"]
    .agg(["min", "max"])
)

for mc_id, dates in campaign_leads.iterrows():

    row_mask = (
        campaigns_clean["mc_id"] ==
        mc_id
    )

    if not row_mask.any():
        continue

    min_lead = dates["min"]
    max_lead = dates["max"]

    if pd.notna(min_lead):
        existing_start = campaigns_clean.loc[
            row_mask,
            "start_date"
        ].iloc[0]

        campaigns_clean.loc[
            row_mask,
            "start_date"
        ] = min(
            existing_start,
            min_lead -
            pd.Timedelta(days=14)
        )

    if pd.notna(max_lead):
        existing_end = campaigns_clean.loc[
            row_mask,
            "end_date"
        ].iloc[0]

        campaigns_clean.loc[
            row_mask,
            "end_date"
        ] = max(
            existing_end,
            max_lead +
            pd.Timedelta(days=14)
        )

In [53]:
students_clean = students_clean[
    [
        "s_id",
        "first_name",
        "last_name",
        "dob",
        "nationality",
        "email",
        "phone_number"
    ]
]


consultants_clean = consultants_clean[
    [
        "c_id",
        "consultant_name",
        "consultant_role",
        "email",
        "phone_number"
    ]
]


campaigns_clean = campaigns_clean[
    [
        "mc_id",
        "campaign_name",
        "platform",
        "start_date",
        "end_date",
        "budget_eur",
        "actual_spend_eur"
    ]
]


journeys_clean = journeys_clean[
    [
        "sj_id",
        "s_id",
        "c_id",
        "mc_id",
        "lead_date",
        "lead_source",
        "target_intake_term",
        "target_intake_year",
        "current_stage",
        "case_status"
    ]
]


applications_clean = applications_clean[
    [
        "app_id",
        "sj_id",
        "university_name",
        "course_name",
        "application_status",
        "application_outcome"
    ]
]


visas_clean = visas_clean[
    [
        "visa_id",
        "sj_id",
        "app_id",
        "visa_status",
        "visa_outcome"
    ]
]




In [54]:
def validate_primary_key(df, key, table_name):

    null_count = df[key].isna().sum()
    duplicate_count = df[key].duplicated().sum()

    print(
        f"{table_name}: "
        f"NULL PK={null_count}, "
        f"duplicate PK={duplicate_count}"
    )

    assert null_count == 0, (
        f"{table_name}.{key} contains NULL values"
    )

    assert duplicate_count == 0, (
        f"{table_name}.{key} contains duplicates"
    )


validate_primary_key(
    students_clean,
    "s_id",
    "students"
)

validate_primary_key(
    consultants_clean,
    "c_id",
    "consultants"
)

validate_primary_key(
    campaigns_clean,
    "mc_id",
    "marketing_campaigns"
)

validate_primary_key(
    journeys_clean,
    "sj_id",
    "student_journeys"
)

validate_primary_key(
    applications_clean,
    "app_id",
    "university_applications"
)

validate_primary_key(
    visas_clean,
    "visa_id",
    "visa_applications"
)


students: NULL PK=0, duplicate PK=0
consultants: NULL PK=0, duplicate PK=0
marketing_campaigns: NULL PK=0, duplicate PK=0
student_journeys: NULL PK=0, duplicate PK=0
university_applications: NULL PK=0, duplicate PK=0
visa_applications: NULL PK=0, duplicate PK=0


In [55]:
def validate_fk(
    child_df,
    child_column,
    parent_df,
    parent_column,
    relationship_name,
    nullable=False
):

    values = child_df[child_column]

    if nullable:
        values = values.dropna()

    invalid = (
        ~values.isin(
            parent_df[parent_column]
        )
    )

    count = invalid.sum()

    print(
        relationship_name,
        "-> invalid:",
        count
    )

    assert count == 0, (
        f"Broken relationship: "
        f"{relationship_name}"
    )

In [56]:
validate_fk(
    journeys_clean,
    "s_id",
    students_clean,
    "s_id",
    "journey -> student"
)

validate_fk(
    journeys_clean,
    "c_id",
    consultants_clean,
    "c_id",
    "journey -> consultant"
)

validate_fk(
    journeys_clean,
    "mc_id",
    campaigns_clean,
    "mc_id",
    "journey -> campaign",
    nullable=True
)

validate_fk(
    applications_clean,
    "sj_id",
    journeys_clean,
    "sj_id",
    "application -> journey"
)

validate_fk(
    visas_clean,
    "sj_id",
    journeys_clean,
    "sj_id",
    "visa -> journey"
)

validate_fk(
    visas_clean,
    "app_id",
    applications_clean,
    "app_id",
    "visa -> university application",
    nullable=True
)


journey -> student -> invalid: 0
journey -> consultant -> invalid: 0
journey -> campaign -> invalid: 0
application -> journey -> invalid: 0
visa -> journey -> invalid: 0
visa -> university application -> invalid: 0


In [57]:
visa_with_app = (
    visas_clean[
        visas_clean["app_id"].notna()
    ]
)

visa_relationship_check = (
    visa_with_app.merge(
        applications_clean[
            ["app_id", "sj_id"]
        ],
        on=[
            "app_id",
            "sj_id"
        ],
        how="left",
        indicator=True
    )
)

invalid_composite_fk = (
    visa_relationship_check["_merge"] !=
    "both"
).sum()

print(
    "Invalid visa application/journey pairs:",
    invalid_composite_fk
)

assert invalid_composite_fk == 0

Invalid visa application/journey pairs: 0


In [58]:
REQUIRED_FIELDS = {
    "students": [
        "s_id",
        "first_name",
        "last_name",
        "dob",
        "nationality",
        "email",
        "phone_number"
    ],

    "consultants": [
        "c_id",
        "consultant_name",
        "consultant_role",
        "email"
    ],

    "marketing_campaigns": [
        "mc_id",
        "campaign_name",
        "platform",
        "start_date"
    ],

    "student_journeys": [
        "sj_id",
        "s_id",
        "c_id",
        "lead_date",
        "lead_source",
        "target_intake_term",
        "target_intake_year",
        "current_stage",
        "case_status"
    ],

    "university_applications": [
        "app_id",
        "sj_id",
        "university_name",
        "course_name",
        "application_status"
    ],

    "visa_applications": [
        "visa_id",
        "sj_id",
        "visa_status"
    ]
}

In [59]:
FINAL_TABLES = {
    "students": students_clean,
    "consultants": consultants_clean,
    "marketing_campaigns": campaigns_clean,
    "student_journeys": journeys_clean,
    "university_applications": applications_clean,
    "visa_applications": visas_clean
}

In [60]:
for table_name, columns in REQUIRED_FIELDS.items():

    print("\n", table_name)

    for column in columns:

        missing = (
            FINAL_TABLES[
                table_name
            ][column]
            .isna()
            .sum()
        )

        print(
            f"{column:25} {missing}"
        )


 students
s_id                      0
first_name                0
last_name                 0
dob                       0
nationality               0
email                     0
phone_number              0

 consultants
c_id                      0
consultant_name           0
consultant_role           0
email                     0

 marketing_campaigns
mc_id                     0
campaign_name             0
platform                  0
start_date                0

 student_journeys
sj_id                     0
s_id                      0
c_id                      0
lead_date                 0
lead_source               0
target_intake_term        0
target_intake_year        0
current_stage             0
case_status               0

 university_applications
app_id                    0
sj_id                     0
university_name           0
course_name               0
application_status        0

 visa_applications
visa_id                   0
sj_id                     0
visa_status         

In [61]:
print(
    "Future leads:",
    (
        journeys_clean["lead_date"] >
        AS_OF_DATE
    ).sum()
)

print(
    "Past target intake years:",
    (
        journeys_clean["target_intake_year"] <
        journeys_clean["lead_date"].dt.year
    ).sum()
)

print(
    "Campaign end before start:",
    (
        campaigns_clean["end_date"] <
        campaigns_clean["start_date"]
    ).sum()
)

print(
    "Invalid visa/application relationships:",
    invalid_composite_fk
)

Future leads: 0
Past target intake years: 0
Campaign end before start: 0
Invalid visa/application relationships: 0


In [62]:
for name, df in FINAL_TABLES.items():

    print(
        "\n",
        "=" * 60,
        "\n",
        name,
        df.shape
    )

    display(df.head())


 students (1000, 7)


,s_id,first_name,last_name,dob,nationality,email,phone_number
0,1,Oralla,Stoter,1995-07-20,Myanmar,oralla.stoter.1@student.example,+95-9-000-000001
1,2,Worden,Dawkins,1998-09-07,Myanmar,worden.dawkins.2@student.example,+95-9-000-000002
2,3,Dale,Gorrie,2005-04-04,Myanmar,dale.gorrie.3@student.example,+95-9-000-000003
3,4,Trudy,Wyne,1997-06-17,Myanmar,trudy.wyne.4@student.example,+95-9-000-000004
4,5,Nap,Ugo,2004-11-16,Myanmar,nap.ugo.5@student.example,+95-9-000-000005



 consultants (5, 5)


,c_id,consultant_name,consultant_role,email,phone_number
0,1,Marius Itshak,Senior Education Consultant,mariusitshak@agency.example,+353-1-555-1000
1,2,Hernando Bamford,Education Consultant,hernandobamford@agency.example,+353-1-555-1001
2,3,Shay Hastie,Admissions Consultant,shayhastie@agency.example,+353-1-555-1002
3,4,Damaris Wilce,Visa Consultant,damariswilce@agency.example,+353-1-555-1003
4,5,Janelle Devonald,Team Lead,janelledevonald@agency.example,+353-1-555-1004



 marketing_campaigns (30, 7)


...

,sj_id,s_id,c_id,mc_id,lead_date,lead_source,target_intake_term,target_intake_year,current_stage,case_status
0,1,1,1,8,2025-03-24,YouTube,January,2027,Visa Submitted,Active
1,2,2,4,<NA>,2026-09-14,Google,May,2028,Closed Lost,Lost
2,3,3,4,<NA>,2026-04-18,Student Referral,May,2028,New Lead,Active
3,4,4,3,14,2026-03-26,YouTube,May,2027,Arrived in Ireland,Completed
4,5,5,3,29,2025-04-09,Referral Campaign,January,2026,Consultation Completed,Active



 university_applications (1118, 6)


,app_id,sj_id,university_name,course_name,application_status,application_outcome
0,1,1,Dublin City University,MSc in Cybersecurity,Offer Issued,Offer
1,2,4,Dublin City University,MSc in Financial Technology,Offer Issued,Offer
2,3,6,Maynooth University,MSc in Data Analytics,Offer Issued,Offer
3,4,7,Technological University Dublin,BSc in Accounting and Finance,Documents Pending,NaN
4,5,8,Griffith College,BA in Digital Media,Documents Pending,NaN



 visa_applications (289, 5)


,visa_id,sj_id,app_id,visa_status,visa_outcome
0,1,1,1,Pending Decision,NaN
1,2,4,2,Approved,Approved
2,3,6,3,Pending Decision,NaN
3,4,9,6,Approved,Approved
4,5,12,8,Preparing,NaN


In [63]:
total_leads = (
    journeys_clean["sj_id"]
    .nunique()
)


SUBMITTED_APPLICATION_STATUSES = [
    "Submitted",
    "Under Review",
    "Offer Issued",
    "Rejected",
    "Withdrawn",
    "Deferred"
]

application_students = (
    applications_clean[
        applications_clean[
            "application_status"
        ].isin(
            SUBMITTED_APPLICATION_STATUSES
        )
    ]["sj_id"]
    .nunique()
)


SUBMITTED_VISA_STATUSES = [
    "Submitted",
    "Pending Decision",
    "Approved",
    "Refused",
    "Withdrawn"
]

visa_students = (
    visas_clean[
        visas_clean[
            "visa_status"
        ].isin(
            SUBMITTED_VISA_STATUSES
        )
    ]["sj_id"]
    .nunique()
)


approved_visas = (
    visas_clean[
        visas_clean["visa_status"] ==
        "Approved"
    ]["sj_id"]
    .nunique()
)


final_visa_decisions = (
    visas_clean[
        visas_clean[
            "visa_status"
        ].isin(
            [
                "Approved",
                "Refused"
            ]
        )
    ]["sj_id"]
    .nunique()
)


successful_students = (
    journeys_clean[
        journeys_clean[
            "current_stage"
        ].isin(
            [
                "Arrived in Ireland",
                "Enrolled"
            ]
        )
    ]["sj_id"]
    .nunique()
)

In [64]:
lead_to_application_rate = (
    application_students /
    total_leads
    if total_leads
    else 0
)

lead_to_visa_rate = (
    visa_students /
    total_leads
    if total_leads
    else 0
)

visa_approval_rate = (
    approved_visas /
    final_visa_decisions
    if final_visa_decisions
    else 0
)

successful_lead_rate = (
    successful_students /
    total_leads
    if total_leads
    else 0
)


print(
    f"Total leads: {total_leads}"
)

print(
    f"Lead → Application: "
    f"{lead_to_application_rate:.2%}"
)

print(
    f"Lead → Visa Submitted: "
    f"{lead_to_visa_rate:.2%}"
)

print(
    f"Visa Approval Rate: "
    f"{visa_approval_rate:.2%}"
)

print(
    f"Successful Lead Rate: "
    f"{successful_lead_rate:.2%}"
)

Total leads: 1000
Lead → Application: 64.20%
Lead → Visa Submitted: 21.50%
Visa Approval Rate: 90.00%
Successful Lead Rate: 6.00%


In [65]:
stage_distribution = (
    journeys_clean[
        "current_stage"
    ]
    .value_counts()
    .rename_axis("stage")
    .reset_index(
        name="students"
    )
)

display(stage_distribution)

,stage,students
0,Application Submitted,147
1,New Lead,123
2,Offer Received,106
3,Visa Submitted,80
4,Closed Lost,79
5,Visa Approved,75
6,Visa Preparation,74
7,Offer Accepted,72
8,Application In Progress,71
9,Consultation Scheduled,65


In [66]:
display(
    journeys_clean[
        "case_status"
    ]
    .value_counts()
    .rename_axis("case_status")
    .reset_index(name="students")
)

,case_status,students
0,Active,861
1,Lost,79
2,Completed,60


In [67]:
for table_name, df in FINAL_TABLES.items():

    output_path = (
        CLEAN_DIR /
        f"{table_name}_clean.csv"
    )

    df.to_csv(
        output_path,
        index=False,
        na_rep="",
        date_format="%Y-%m-%d"
    )

    print(
        "Saved:",
        output_path
    )

Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\students_clean.csv
Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\consultants_clean.csv
Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\marketing_campaigns_clean.csv
Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\student_journeys_clean.csv
Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\university_applications_clean.csv
Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\visa_applications_clean.csv
